# Build STAC Catalog for FVCOM GOM3 Icechunk Store

Creates a static STAC catalog with a single item pointing to the Icechunk repository
created by `1_create_virtual_icechunk.ipynb`, then uploads the catalog JSON files to S3.

Uses:
- `pystac` for catalog/item construction
- `xstac` for datacube extension metadata
- `icechunk` to open the store and read the snapshot ID

In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path

import icechunk
import pystac
import s3fs
import xarray as xr
import xstac
from dotenv import load_dotenv

In [ ]:
load_dotenv(os.path.expanduser('~/dotenv/umassd-fvcom.yml'))

## Open the Icechunk store (read-only, anonymous)

In [ ]:
bucket = 'umassd-fvcom'
prefix = 'gom3/hindcast/icechunk/gom3.icechunk'
region = 'us-east-1'
store_uri = f's3://{bucket}/{prefix}'

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix='s3://umassd-fvcom/',
        store=icechunk.s3_store(region=region, anonymous=True),
    ),
)

storage = icechunk.s3_storage(
    bucket=bucket,
    prefix=prefix,
    region=region,
    anonymous=True,
)

credentials = icechunk.containers_credentials({
    's3://umassd-fvcom/': icechunk.s3_credentials(anonymous=True)
})
repo = icechunk.Repository.open(
    storage,
    config,
    authorize_virtual_chunk_access=credentials,
)
session = repo.readonly_session('main')
snapshot_id = session.snapshot_id
print(f'Snapshot ID: {snapshot_id}')

In [ ]:
ds = xr.open_zarr(session.store, consolidated=False, chunks=None)
ds

## Extract spatial and temporal extent

In [ ]:
lon_min = float(ds['lon'].min())
lon_max = float(ds['lon'].max())
lat_min = float(ds['lat'].min())
lat_max = float(ds['lat'].max())
bbox = [lon_min, lat_min, lon_max, lat_max]

time_min = datetime.fromisoformat(str(ds['time'].min().values.astype('datetime64[s]')))
time_max = datetime.fromisoformat(str(ds['time'].max().values.astype('datetime64[s]')))

print(f'Bounding box: {bbox}')
print(f'Time range:   {time_min} — {time_max}')

## Build the STAC item

In [ ]:
ICECHUNK_MEDIA_TYPE = 'application/vnd.zarr+icechunk'
STAC_EXTENSIONS = [
    'https://stac-extensions.github.io/storage/v2.0.0/schema.json',
    'https://stac-extensions.github.io/virtual-assets/v1.0.0/schema.json',
    'https://stac-extensions.github.io/zarr/v1.1.0/schema.json',
    'https://stac-extensions.github.io/version/v1.2.0/schema.json',
    'https://stac-extensions.github.io/datacube/v2.2.0/schema.json',
]

def bbox_to_geometry(bbox):
    l, b, r, t = bbox
    return {'type': 'Polygon', 'coordinates': [[[l, b], [l, t], [r, t], [r, b], [l, b]]]}

item = pystac.Item(
    id='umassd-fvcom-gom3-hindcast',
    geometry=bbox_to_geometry(bbox),
    bbox=bbox,
    datetime=None,
    start_datetime=time_min,
    end_datetime=time_max,
    properties={
        'title': 'UMASSD FVCOM GOM3 Hindcast',
        'description': (
            'Gulf of Maine 3 (GOM3) ocean hindcast from the UMASSD FVCOM model, '
            '1978–2016. Virtual Icechunk store referencing 468 uncompressed NetCDF3 '
            'source files on AWS Open Data. Vertical variables subchunked to 1 '
            'sigma layer per chunk.'
        ),
    },
    stac_extensions=STAC_EXTENSIONS,
)

# Storage scheme declaration (required by the storage extension)
item.extra_fields['storage:schemes'] = {
    f'aws-s3-{bucket}': {
        'type': 'aws-s3',
        'bucket': bucket,
        'region': region,
        'anonymous': True,
    }
}

# Primary data asset keyed by snapshot ID
item.add_asset(
    f'data@{snapshot_id}',
    pystac.Asset(
        href=store_uri,
        media_type=ICECHUNK_MEDIA_TYPE,
        roles=['data', 'references'],
        extra_fields={
            'icechunk:snapshot_id': snapshot_id,
            'version': snapshot_id,
            'storage:refs': [f'aws-s3-{bucket}'],
        },
    ),
)

In [ ]:
# Populate datacube extension via xstac
item = xstac.xarray_to_stac(
    ds,
    item,
    temporal_dimension='time',
    x_dimension='node',
    y_dimension='node',
    reference_system=4326,
    validate=False,
)
item.to_dict()

## Build the STAC catalog

In [ ]:
catalog = pystac.Catalog(
    id='umassd-fvcom-gom3',
    description=(
        'UMASSD FVCOM GOM3 hindcast — virtual Icechunk store on AWS S3 '
        'referencing 468 uncompressed NetCDF3 source files.'
    ),
    catalog_type=pystac.CatalogType.SELF_CONTAINED,
)
catalog.add_item(item)

## Save catalog locally and upload to S3

In [ ]:
output_dir = Path('./stac_catalog')
output_dir.mkdir(exist_ok=True)

catalog.normalize_hrefs(str(output_dir))
catalog.save()

print('Local catalog files:')
for p in sorted(output_dir.rglob('*.json')):
    print(' ', p)

In [ ]:
# Upload catalog JSON files to S3
stac_s3_prefix = 'gom3/hindcast/stac'

fs_s3 = s3fs.S3FileSystem(
    key=os.environ['AWS_ACCESS_KEY_ID'],
    secret=os.environ['AWS_SECRET_ACCESS_KEY'],
)

for local_path in sorted(output_dir.rglob('*.json')):
    s3_key = f'{bucket}/{stac_s3_prefix}/{local_path.relative_to(output_dir)}'
    fs_s3.put(str(local_path), s3_key)
    print(f'Uploaded s3://{s3_key}')

In [ ]:
# Print the catalog JSON for review
with open(output_dir / 'catalog.json') as f:
    print(json.dumps(json.load(f), indent=2))